# This notebook shows how to compute anisotropic magnetic exchange constants from DFT

**Cells below are not executed here**: each component of the exchange tensor costs four
constrained non-collinear SCF runs, so the full $3\times3$ tensor is 36 runs of a 16-atom
all-electron cell -- hours, not minutes. Run it yourself with
`jupyter nbconvert --execute --inplace notebooks/18_exchange_constants.ipynb`, or start with
`components="diagonal"` (12 runs). Same reason `03_phonon_dispersion_and_dos.ipynb` is left
unexecuted.

### Import the necessary libraries

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from ase.build import bulk

from elkpy.structure import Structure
from elkpy.parsers import exchange
from elkpy import exchange as exchange_module

### The spin Hamiltonian

$$ H=\sum_{i<j}\mathbf{S}_i\cdot\mathbf{J}_{ij}\cdot\mathbf{S}_j+\sum_i\mathbf{S}_i\cdot\mathbf{A}_{ii}\cdot\mathbf{S}_i $$

$\mathbf{J}_{ij}$ is a general $3\times3$ matrix, each pair counted once, spins of length $S$;
$J<0$ is ferromagnetic. Its isotropic part is Heisenberg exchange, its traceless symmetric part
is anisotropic exchange (the Kitaev $K$ and $\Gamma$ terms on a honeycomb), and its antisymmetric
part is the Dzyaloshinskii-Moriya vector.

### The four-state method

$$ J_{ij}^{\alpha\beta}=\frac{E_1+E_4-E_2-E_3}{4S^2m_{ij}} $$

$E_{1\ldots4}$ are total energies of four constrained states with $\mathbf{S}_i=\pm S\hat{\alpha}$
and $\mathbf{S}_j=\pm S\hat{\beta}$, every other spin held along the third axis. Single-ion
anisotropy is quadratic in spin and so cancels; $m_{ij}$ counts how many copies of the bond a
single flip turns over, periodic images included.

In [ ]:
# NiO: rocksalt antiferromagnet, dominant coupling is the 180-degree Ni-O-Ni superexchange
structure = Structure.from_ase(bulk("NiO", "rocksalt", a=4.17, cubic=False) * (2, 2, 2))
calc = structure.get_calculation(
    "_scratch/nio", xc="PW", spinpol=True, ngridk=(2, 2, 2), rgkmax=6.5,   # reduced basis: cost scales steeply with rgkmax
    # FLL double counting, U = 6 eV and J = 1 eV on the Ni d shell, in Hartree
    extra_blocks={"dft+u": [(1, 1), (1, 2, 0.2205, 0.0367)], "nempty": [20]},
)

### Picking the pair, and checking the supercell is big enough

Flipping site $j$ flips all its periodic images, so the energy difference measures
$\sum_{\mathbf{T}}J(i,j+\mathbf{T})$. That is $m_{ij}$ copies of one bond only if every
contributing image sits at the same distance; otherwise the result mixes neighbour shells
irrecoverably.

In [ ]:
avec = np.array(structure.avec)
nickel = exchange_module.magnetic_indices(structure, "Ni")
pos = {k: exchange_module.atom_cartesian(structure, k) for k in nickel}
# the J2 partner sits at the cubic lattice constant, 4.17 A = 7.88 Bohr
i = nickel[0]
j = min(nickel[1:], key=lambda k: abs(exchange.image_distances(avec, pos[i], pos[k])[0] - 7.88))
# raises if a periodic image inside the retained range sits at a different distance
exchange.check_supercell(avec, pos[i], pos[j], shell_cutoff=9.45)

### The exchange tensor

Nine components, 36 constrained SCF runs, threaded over configurations.

In [ ]:
result = calc.get_exchange_tensor(
    i, j, magnetic="Ni",      # spectators must be constrained too, or they are not spectators
    spin=1.0,                 # Ni(II) is S = 1: this sets the convention, not just a scale
    components="all", workers=6,
)
tensor = result["tensor"]     # meV
result["isotropic"], result["dm"]

### What the tensor says

Without spin-orbit coupling nothing ties spin to the lattice, so the energy is invariant under
a global spin rotation and the tensor must be exactly $J_{\rm iso}\mathbb{1}$ -- the off-diagonal
structure below is the numerical noise floor of four summed total energies. Switching
`spinorb=True` on the `Calculation` turns on the anisotropy; the Dzyaloshinskii-Moriya part stays
zero here either way, because the bridging oxygen at the bond midpoint is an inversion centre.

In [ ]:
fig, (ax, bx) = plt.subplots(1, 2, figsize=(9, 3.6))

im = ax.matshow(tensor, cmap="RdBu_r", vmin=-abs(tensor).max(), vmax=abs(tensor).max())
for a in range(3):
    for b in range(3):
        ax.text(b, a, f"{tensor[a, b]:.2f}", ha="center", va="center", fontsize=9)
ax.set_xticks(range(3), ["x", "y", "z"])
ax.set_yticks(range(3), ["x", "y", "z"])
ax.set_title(r"$J_{ij}^{lphaeta}$ (meV)")
fig.colorbar(im, ax=ax)

parts = {"isotropic": abs(result["isotropic"]),
         "symmetric": np.abs(result["symmetric"]).max(),
         "DM": np.linalg.norm(result["dm"])}
bx.bar(list(parts), list(parts.values()), color=["#4C72B0", "#DD8452", "#55A868"])
bx.set_yscale("log")
bx.set_ylabel("magnitude (meV)")
bx.set_title("decomposition")
plt.tight_layout()
plt.show()

### Kitaev magnets

For a honeycomb lattice with edge-sharing octahedra, rotate the tensor into the cubic
octahedral frame first; the $z$-labelled bond then has the Rau-Lee-Kee form

$$ \mathbf{J}_Z=\begin{pmatrix} J & \Gamma & \Gamma' \\ \Gamma & J & \Gamma' \\ \Gamma' & \Gamma' & J+K \end{pmatrix} $$

with $K$ the Kitaev term. `residual` reports how far the measured tensor is from that ideal
form, which only bonds with full $C_{2h}$ midpoint symmetry are forced into.

In [ ]:
cubic = exchange.rotate_tensor(tensor, exchange.honeycomb_cubic_frame())
exchange.kitaev_parameters(cubic)